# Industry Project: Developing a Prompt Like Software (Eval-Driven Prompt Engineering)

> **Who is this notebook for?** Someone who has written a few prompts by trial and error ("let me tweak this sentence and look at two outputs") and wants to know how professional teams make prompt changes **measurable, reviewable and reversible**.
> No API key is needed: everything runs offline on a CPU with a small open model. Almost every line of code is commented.

---

### How this notebook is different from the others in this folder

| `01_basics` … `04_adversarial` | This notebook |
|---|---|
| Explain **techniques**: roles, few-shot, chain-of-thought, structured output, injection | Shows the **process** that decides which technique to keep |
| One prompt, one example output (often needs an API key) | A **labelled eval set**, every prompt version **scored** on it (runs offline) |
| "Few-shot helps" | *Measured*: did few-shot help **here**? (spoiler: not everywhere) |
| Prompts live in the code | Prompts are **versioned artifacts** in a registry, with a **regression gate** before promotion |

The companion notebook `../08_LLM/20_industry_llm_application_patterns.ipynb` covers what you build *around* one LLM call (validation, retries, caching, fallbacks, guardrails). Here we focus on the **development loop** for the prompt itself.

---

### The business problem

**Pocketly** (a fictional budgeting app, ~40,000 in-app feedback messages per month) wants every message turned into a ticket automatically:

| Field | Allowed values | Used for |
|---|---|---|
| `category` | `bug`, `feature_request`, `billing`, `account_access`, `praise` | routing to the right team |
| `platform` | `ios`, `android`, `web`, or `null` | assigning the right engineers |
| `app_version` | e.g. `"5.3.0"`, or `null` | finding which release broke something |

A wrong `category` sends a locked-out customer to the product team instead of support; that costs hours and sometimes a customer. The product manager asks: *"Which prompt do we ship, and how do we know the next change does not make it worse?"*

### The workflow we follow (the loop real teams run)

```
1. Define task + success criteria ─► 2. Build a labelled eval set ─► 3. Baseline prompt v1 ─► 4. Measure
        ▲                                                                                    │
        │                                                                                    ▼
 9. Registry: promote / roll back ◄─ 8. Regression gate ◄─ 7. Compare versions ◄─ 6. Iterate ◄─ 5. Error analysis
```

### Table of Contents
1. Setup: imports and one central config
2. Define the task and the success criteria (before writing any prompt)
3. Build a small labelled eval set
4. The LLM client (local model, optional API)
5. The evaluation harness: parse, validate, score, time
6. Prompt v1: the day-one baseline
7. Prompt v2: clear instructions + output schema
8. Error analysis: look at the failures, then name them
9. Prompt v3: add few-shot examples
10. Prompt v4: targeted fixes from the error analysis
11. Prompt v5: move what code does better out of the prompt
12. Compare all versions (with honest uncertainty)
13. Regression gate: block prompts that make things worse
14. Prompt registry: versioning, promotion, rollback
15. Launch readiness and the official prompt-engineering tips
16. Summary, exercises and resources

---
## 1. Setup: imports and one central config

In [1]:
# ---- Standard library ----
import hashlib                       # fingerprint of a prompt version (like a git commit hash)
import json
import logging
import os
import re                            # regex for parsing outputs and for the app_version extractor
import tempfile                      # the prompt registry is saved in a temp folder, never in the repo
import time
import warnings
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path

# If the local model is already downloaded, tell Hugging Face not to contact the Hub at all (must happen
# BEFORE transformers is imported). On a fresh machine the model (~1 GB) is downloaded once and cached.
_hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
if (_hf_home / "hub" / "models--Qwen--Qwen2.5-0.5B-Instruct").exists():
    _hf_hub = os.environ.get("HF_HUB_CACHE") or os.path.join(os.environ.get("HF_HOME") or os.path.join(os.path.expanduser("~"), ".cache", "huggingface"), "hub")
    if all(os.path.isdir(os.path.join(_hf_hub, "models--" + m.replace("/", "--"))) for m in ['Qwen/Qwen2.5-0.5B-Instruct']):
        os.environ.setdefault("HF_HUB_OFFLINE", "1")   # model already downloaded: never touch the internet
    # (if it is not downloaded yet, the first run downloads it once from Hugging Face)

warnings.filterwarnings("ignore")      # set before importing transformers to keep the output clean

# ---- Third-party ----
import numpy as np
import pandas as pd
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)
log = logging.getLogger("prompt-dev")
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)


@dataclass(frozen=True)
class Config:
    max_new_tokens: int = 40           # our JSON answer needs ~25 tokens; the cap also limits cost and rambling
    bootstrap_rounds: int = 2000       # resamples for confidence intervals
    seed: int = 42
    budget_usd_per_1k_msgs: float = 0.10   # max LLM cost per 1,000 messages (input + output tokens)


CFG = Config()
rng = np.random.default_rng(CFG.seed)
REGISTRY_DIR = Path(tempfile.mkdtemp(prefix="prompt_registry_"))   # real project: prompts/ folder in git
print(CFG)

Config(max_new_tokens=40, bootstrap_rounds=2000, seed=42, budget_usd_per_1k_msgs=0.1)


---
## 2. Define the task and the success criteria (before writing any prompt)

Both Anthropic's and OpenAI's prompt-engineering guides start in the same place: **before** you engineer a prompt you need (1) clear success criteria and (2) a way to test against them. Anthropic suggests criteria that are *specific, measurable, achievable and relevant*. Without them, "the new prompt feels better" wins every discussion, and nobody notices when a change quietly breaks something.

Our criteria, agreed with the product manager **before** prompting:

In [2]:
SUCCESS_CRITERIA = {
    # metric name          (target, why)
    "category_accuracy":  (0.90, "routing errors cost hours; 9 in 10 must go to the right team"),
    "platform_accuracy":  (0.90, "wrong platform = wrong engineers"),
    "version_accuracy":   (0.95, "release attribution for bugs"),
    "format_valid_rate":  (0.98, "the ticket system rejects anything that is not valid JSON with the schema"),
    "cost_usd_per_1k":    (CFG.budget_usd_per_1k_msgs, "max LLM spend per 1,000 messages (lower is better)"),
}
pd.DataFrame([{"metric": k, "target": v[0], "why": v[1]} for k, v in SUCCESS_CRITERIA.items()])

,metric,target,why
0,category_accuracy,0.90,routing errors cost hours; 9 in 10 must go to the right team
1,platform_accuracy,0.90,wrong platform = wrong engineers
2,version_accuracy,0.95,release attribution for bugs
3,format_valid_rate,0.98,the ticket system rejects anything that is not valid JSON with the...
4,cost_usd_per_1k,0.10,"max LLM spend per 1,000 messages (lower is better)"


We also write a **labelling guideline**: if two humans would label a message differently, the prompt cannot be "right" either. Ambiguity in the labels becomes noise in every metric. The guideline (which later goes almost word-for-word into the prompt):

- **account_access**: logging in, passwords, SMS/2FA codes, Face ID, locked accounts *(checked first)*
- **billing**: charges, payments, subscriptions, refunds, prices *(even if a bug caused the wrong charge)*
- **bug**: something broken, crashing, showing wrong data or not loading
- **feature_request**: asks for something new *(even if the message also praises the app)*
- **praise**: only positive feedback, nothing to fix or add
- `platform`: only if a device, OS or browser is named; `app_version`: only if a version number is written.

---
## 3. Build a small labelled eval set

The eval set is the most valuable artifact of the project, more valuable than any single prompt. Good practice:

- Sample **real** messages (here: realistic examples written by the support team), labelled by the guideline.
- Cover **every class** and the **known hard cases** (billing caused by a bug, praise plus a request, a version number in praise, Face ID).
- Mark a few **must-pass** cases: mistakes that would be embarrassing or costly in production.
- Keep few-shot examples for the prompt **separate** from the eval set (otherwise you test on what you taught).

15 messages is tiny: enough to find big problems quickly (Hamel Husain: "look at your data"), far too small to measure small differences. Section 12 shows exactly how uncertain the numbers are; real teams grow the set to hundreds of cases over time, mostly from production failures.

In [3]:
EVAL_SET = pd.DataFrame([
    # id,   message,                                                                                  category,       platform,  version, tags
    ("F01", "App crashes every time I open the Budgets tab. iPhone 14, app version 5.2.1.",            "bug",            "ios",     "5.2.1", "easy"),
    ("F02", "Would love a dark mode for the web dashboard!",                                           "feature_request", "web",     None,    "easy"),
    ("F03", "I was charged $9.99 twice this month for Premium. Please fix.",                           "billing",         None,      None,    "must_pass"),
    ("F04", "Can't log in since yesterday, the SMS code never arrives. Galaxy S23.",                   "account_access",  "android", None,    "must_pass, device-name"),
    ("F05", "Best budgeting app I've ever used, thank you team!",                                      "praise",          None,      None,    "easy"),
    ("F06", "After updating to 5.3.0 on my Pixel 8 the balance shows 0.00 for all accounts.",          "bug",             "android", "5.3.0", "device-name"),
    ("F07", "Please add the option to export transactions to CSV.",                                    "feature_request", None,      None,    "easy"),
    ("F08", "The app froze during the upgrade payment and now I've been charged but I don't have Premium. iOS 18, version 5.2.0",
                                                                                                        "billing",         "ios",     "5.2.0", "hard: bug caused billing"),
    ("F09", "My account is locked after 3 wrong passwords and the reset email link is expired.",       "account_access",  None,      None,    "must_pass"),
    ("F10", "Love the new charts in 5.3.0 on Android, super clear.",                                   "praise",          "android", "5.3.0", "hard: version in praise"),
    ("F11", "Charts don't load in Chrome, just a spinning wheel.",                                     "bug",             "web",     None,    "browser-name"),
    ("F12", "Great app but it would be even better with shared budgets for couples.",                 "feature_request", None,      None,    "hard: praise + request"),
    ("F13", "Why did my subscription renew at the new price without any notice?",                      "billing",         None,      None,    "easy"),
    ("F14", "Face ID login stopped working on my iPad after the update to 5.3.1.",                     "account_access",  "ios",     "5.3.1", "hard: login vs bug"),
    ("F15", "Thanks for fixing the sync issue so quickly in 5.3.1, you rock!",                         "praise",          None,      "5.3.1", "hard: 'fixing' sounds like bug"),
], columns=["id", "message", "category", "platform", "app_version", "tags"])
FIELDS = ["category", "platform", "app_version"]
MUST_PASS = EVAL_SET.loc[EVAL_SET["tags"].str.contains("must_pass"), "id"].tolist()

print(EVAL_SET["category"].value_counts().to_string(), "\nmust-pass cases:", MUST_PASS)
EVAL_SET

category
bug                3
feature_request    3
billing            3
account_access     3
praise             3 
must-pass cases: ['F03', 'F04', 'F09']


,id,message,category,platform,app_version,tags
0,F01,"App crashes every time I open the Budgets tab. iPhone 14, app vers...",bug,ios,5.2.1,easy
1,F02,Would love a dark mode for the web dashboard!,feature_request,web,None,easy
2,F03,I was charged $9.99 twice this month for Premium. Please fix.,billing,None,None,must_pass
3,F04,"Can't log in since yesterday, the SMS code never arrives. Galaxy S23.",account_access,android,None,"must_pass, device-name"
4,F05,"Best budgeting app I've ever used, thank you team!",praise,None,None,easy
5,F06,After updating to 5.3.0 on my Pixel 8 the balance shows 0.00 for a...,bug,android,5.3.0,device-name
6,F07,Please add the option to export transactions to CSV.,feature_request,None,None,easy
7,F08,The app froze during the upgrade payment and now I've been charged...,billing,ios,5.2.0,hard: bug caused billing
8,F09,My account is locked after 3 wrong passwords and the reset email l...,account_access,None,None,must_pass
9,F10,"Love the new charts in 5.3.0 on Android, super clear.",praise,android,5.3.0,hard: version in praise


### The `LLMClient`: one small wrapper around "whatever model we use"

In a company you almost never call a model SDK directly from 50 places in the code. You write **one thin wrapper** so that:

1. switching provider/model is a **config change**, not a rewrite,
2. every call is **measured the same way** (tokens, latency, cost),
3. retries, timeouts, caching and logging can be added in **one place**.

Our wrapper picks a backend in this order:

| Order | Backend | When |
|---|---|---|
| 1 | Anthropic API (`claude-haiku-5-5` by default, `claude-sonnet-5-5` for harder tasks) | `ANTHROPIC_API_KEY` is set **and** the `anthropic` package is installed |
| 2 | OpenAI API (model name from `OPENAI_MODEL`) | `OPENAI_API_KEY` is set **and** the `openai` package is installed |
| 3 | **Local model** `Qwen/Qwen2.5-0.5B-Instruct` via 🤗 `transformers` on CPU | otherwise (this is what produced the outputs you see) |

> ⚠️ **Honesty note.** The local model has **0.5 billion** parameters. Production models are hundreds of times larger. It will make mistakes that a modern API model would not make. That is fine for this notebook: **the engineering patterns are the point**, and a weak model actually makes it easier to see *why* those patterns (validation, retries, fallbacks, evaluation) exist.

---
## 4. The LLM client (local model, optional API)

We reuse the same `LLMClient` as the LLM industry notebooks, and add one method for evaluation runs: `complete_batch()` sends **many** messages with the same prompt. For an API this is a loop (or the providers' **Batch APIs**, which are cheaper for offline evals because results may take minutes to hours); for the local model we put the whole eval set through the model **in one batch**, which is much faster on a CPU. Few-shot examples are sent as real *user → assistant* message pairs, the format both providers recommend.

In [4]:
import os
import time
from dataclasses import dataclass, field

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_num_threads(2)          # few threads: faster on a busy shared CPU (raise it on your own machine)
torch.manual_seed(0)              # reproducibility (we also use greedy decoding below)

# Prices in US$ per 1 million tokens (input, output). Check the provider's pricing page before relying on them.
PRICE_PER_MTOK = {
    "claude-sonnet-5-5": (2.00, 10.00),
    "claude-haiku-5-5": (0.10, 0.50),
    "local": (0.0, 0.0),           # runs on our own CPU: no per-token bill (but hardware and ops are not free)
}


@dataclass
class LLMResponse:
    """Everything we want to know about ONE model call (the unit we log, cache and evaluate)."""
    text: str
    model: str
    provider: str
    input_tokens: int
    output_tokens: int
    latency_s: float
    cost_usd: float = 0.0
    cached: bool = False


class LLMClient:
    """Tiny provider-agnostic chat client: Anthropic -> OpenAI -> local transformers model."""

    def __init__(self, local_model: str = "Qwen/Qwen2.5-0.5B-Instruct", prefer: str | None = None,
                 anthropic_model: str = "claude-haiku-5-5", openai_model: str | None = None):
        self.local_model_name = local_model
        self.anthropic_model = anthropic_model
        self.openai_model = openai_model or os.getenv("OPENAI_MODEL", "gpt-5-mini")
        self.provider = prefer or self._detect_provider()
        self._tok = self._model = None                     # local model is loaded lazily (only if needed)
        if self.provider == "local":
            self._load_local()

    @staticmethod
    def _detect_provider() -> str:
        """Use a paid API only if a key AND the SDK are available; otherwise stay fully offline."""
        try:
            if os.getenv("ANTHROPIC_API_KEY"):
                import anthropic  # noqa: F401
                return "anthropic"
        except ImportError:
            pass
        try:
            if os.getenv("OPENAI_API_KEY"):
                import openai  # noqa: F401
                return "openai"
        except ImportError:
            pass
        return "local"

    def _load_local(self):
        t0 = time.perf_counter()
        self._tok = AutoTokenizer.from_pretrained(self.local_model_name)
        self._model = AutoModelForCausalLM.from_pretrained(self.local_model_name, dtype=torch.float32)
        self._model.eval()                                  # inference mode (no dropout)
        print(f"Loaded local model {self.local_model_name} in {time.perf_counter() - t0:.1f}s")

    @property
    def model_name(self) -> str:
        return {"anthropic": self.anthropic_model, "openai": self.openai_model}.get(self.provider, "local")

    def count_tokens(self, text: str) -> int:
        """Token count with the local tokenizer (API providers return exact counts themselves)."""
        if self._tok is None:
            return max(1, len(text) // 4)                   # rough rule of thumb: ~4 characters per token
        return len(self._tok.encode(text))

    def chat(self, system: str, user: str, max_new_tokens: int = 128) -> LLMResponse:
        """One chat turn. temperature=0 / greedy decoding => (almost) deterministic answers."""
        t0 = time.perf_counter()
        if self.provider == "anthropic":
            import anthropic
            msg = anthropic.Anthropic().messages.create(
                model=self.anthropic_model, max_tokens=max_new_tokens, system=system,
                messages=[{"role": "user", "content": user}])
            text = "".join(b.text for b in msg.content if b.type == "text")
            n_in, n_out = msg.usage.input_tokens, msg.usage.output_tokens
        elif self.provider == "openai":
            from openai import OpenAI
            resp = OpenAI().chat.completions.create(
                model=self.openai_model, max_completion_tokens=max_new_tokens,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
            text = resp.choices[0].message.content or ""
            n_in, n_out = resp.usage.prompt_tokens, resp.usage.completion_tokens
        else:
            messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
            prompt = self._tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self._tok(prompt, return_tensors="pt")
            with torch.no_grad():                           # no gradients needed for inference
                out = self._model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                           pad_token_id=self._tok.eos_token_id)
            new_tokens = out[0, inputs["input_ids"].shape[1]:]          # keep only the generated part
            text = self._tok.decode(new_tokens, skip_special_tokens=True)
            n_in, n_out = int(inputs["input_ids"].shape[1]), int(len(new_tokens))
        price_in, price_out = PRICE_PER_MTOK.get(self.model_name, (0.0, 0.0))
        return LLMResponse(text=text.strip(), model=self.model_name, provider=self.provider,
                           input_tokens=n_in, output_tokens=n_out,
                           latency_s=round(time.perf_counter() - t0, 3),
                           cost_usd=(n_in * price_in + n_out * price_out) / 1e6)


def api_cost_estimate(input_tokens: int, output_tokens: int, model: str = "claude-haiku-5-5") -> float:
    """What the same call WOULD cost on a paid API (useful for budgeting before you switch)."""
    price_in, price_out = PRICE_PER_MTOK[model]
    return (input_tokens * price_in + output_tokens * price_out) / 1e6

In [5]:
class EvalLLMClient(LLMClient):
    """LLMClient + a batch method that supports few-shot examples as chat turns."""

    def _messages(self, examples, user):
        msgs = []
        for ex_user, ex_answer in examples:                     # few-shot = earlier turns of the conversation
            msgs += [{"role": "user", "content": ex_user}, {"role": "assistant", "content": ex_answer}]
        return msgs + [{"role": "user", "content": user}]

    def complete_batch(self, system: str, users: list[str], examples=(), max_new_tokens: int = 40) -> list[LLMResponse]:
        if self.provider != "local":                            # APIs: one call per message (or the Batch API)
            out = []
            for u in users:
                t0 = time.perf_counter()
                if self.provider == "anthropic":
                    import anthropic
                    msg = anthropic.Anthropic().messages.create(model=self.anthropic_model, max_tokens=max_new_tokens,
                                                                system=system, messages=self._messages(examples, u))
                    text, n_in, n_out = msg.content[0].text, msg.usage.input_tokens, msg.usage.output_tokens
                else:
                    from openai import OpenAI
                    resp = OpenAI().chat.completions.create(
                        model=self.openai_model, max_completion_tokens=max_new_tokens,
                        messages=[{"role": "system", "content": system}] + self._messages(examples, u))
                    text, n_in, n_out = resp.choices[0].message.content, resp.usage.prompt_tokens, resp.usage.completion_tokens
                out.append(LLMResponse(text.strip(), self.model_name, self.provider, n_in, n_out,
                                       round(time.perf_counter() - t0, 3)))
            return out
        # Local model: left-pad all prompts and generate them together in ONE forward pass per token
        self._tok.padding_side = "left"
        prompts = [self._tok.apply_chat_template([{"role": "system", "content": system}] + self._messages(examples, u),
                                                 tokenize=False, add_generation_prompt=True) for u in users]
        t0 = time.perf_counter()
        batch = self._tok(prompts, return_tensors="pt", padding=True)
        with torch.no_grad():
            out = self._model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                                       pad_token_id=self._tok.eos_token_id)
        seconds_each = (time.perf_counter() - t0) / len(users)          # amortised time per message
        results = []
        for i, row in enumerate(out):
            new = row[batch["input_ids"].shape[1]:]
            text = self._tok.decode(new, skip_special_tokens=True)
            n_in = int(batch["attention_mask"][i].sum())                 # real prompt tokens (without padding)
            n_out = int((new != self._tok.eos_token_id).sum())
            results.append(LLMResponse(text.strip(), "local", "local", n_in, n_out, round(seconds_each, 3)))
        return results


llm = EvalLLMClient()
print("Provider:", llm.provider, "| model:", llm.model_name)

Loaded local model Qwen/Qwen2.5-0.5B-Instruct in 1.2s
Provider: local | model: local


---
## 5. The evaluation harness: parse, validate, score, time

A prompt version in industry is more than the text. It is a **spec**: system prompt + user template + few-shot examples + model + parameters (+ any post-processing code). Change any of them and the behaviour changes, so the **fingerprint** (hash) covers all of them. OpenAI's guide also recommends pinning production apps to a fixed **model snapshot** for the same reason.

The harness does four things for every message:
1. **Run** the prompt (with a cache keyed by fingerprint + message, so re-running a notebook cell does not pay twice).
2. **Parse** tolerantly (strip code fences, find the JSON object) and **validate** strictly against the schema.
3. **Score** each field against the label.
4. Record **cost and time** (tokens × price; seconds per message).

In [6]:
ALLOWED = {"category": {"bug", "feature_request", "billing", "account_access", "praise"},
           "platform": {"ios", "android", "web", None}}
VERSION_RE = re.compile(r"^\d+\.\d+(\.\d+)?$")


@dataclass
class PromptSpec:
    """Everything that determines the model's behaviour. Hashing all of it gives the version fingerprint."""
    name: str
    version: str
    system: str
    user_template: str = "<feedback>\n{message}\n</feedback>"
    examples: list = field(default_factory=list)          # [(user, assistant), ...]
    model: str = "local:Qwen/Qwen2.5-0.5B-Instruct"
    max_new_tokens: int = CFG.max_new_tokens
    postprocess: str = "none"                              # name of a deterministic code step (Section 11)
    changelog: str = ""

    @property
    def fingerprint(self) -> str:
        """Hash of the WHOLE spec: changes whenever anything that affects the output changes."""
        payload = json.dumps({k: v for k, v in asdict(self).items() if k not in ("version", "changelog")}, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()[:12]

    @property
    def llm_fingerprint(self) -> str:
        """Hash of only what the MODEL sees: used as cache key, so code-only changes reuse model outputs."""
        payload = json.dumps([self.system, self.user_template, self.examples, self.model, self.max_new_tokens])
        return hashlib.sha256(payload.encode()).hexdigest()[:12]


def parse_output(text: str) -> tuple[dict | None, list[str]]:
    """Tolerant parsing, strict validation. Returns (fields or None, list of format problems)."""
    problems = []
    if text.startswith("```"):
        problems.append("code fence")                    # extra text around the JSON = format violation
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        return None, problems + ["no JSON object"]
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None, problems + ["invalid JSON"]
    if not isinstance(data, dict):
        return None, problems + ["not an object"]
    if set(data) != set(FIELDS):
        problems.append(f"keys {sorted(data)}")
    out = {}
    for f in FIELDS:
        value = data.get(f)
        if f in ALLOWED and value not in ALLOWED[f]:
            problems.append(f"{f}={value!r} not allowed")       # e.g. "null" as a string, "iOS", "Bug Report"
        if f == "app_version" and value is not None and not (isinstance(value, str) and VERSION_RE.match(value)):
            problems.append(f"app_version={value!r} not allowed")
        out[f] = value
    return out, problems


POSTPROCESSORS = {"none": lambda fields, message: fields}      # filled in Section 11
RESULT_CACHE: dict = {}                                        # (llm_fingerprint, message) -> LLMResponse


def run_eval(spec: PromptSpec, eval_set: pd.DataFrame = EVAL_SET) -> pd.DataFrame:
    """Run one prompt version on the eval set and return one scored row per message."""
    todo = [m for m in eval_set["message"] if (spec.llm_fingerprint, m) not in RESULT_CACHE]
    if todo:
        log.info("Running %s (%s) on %d messages ...", spec.version, spec.fingerprint, len(todo))
        responses = llm.complete_batch(spec.system, [spec.user_template.format(message=m) for m in todo],
                                       examples=spec.examples, max_new_tokens=spec.max_new_tokens)
        RESULT_CACHE.update({(spec.llm_fingerprint, m): r for m, r in zip(todo, responses)})
    else:
        log.info("%s: all %d model outputs found in the cache (same prompt, same model)", spec.version, len(eval_set))
    rows = []
    for _, case in eval_set.iterrows():
        resp = RESULT_CACHE[(spec.llm_fingerprint, case["message"])]
        fields, problems = parse_output(resp.text)
        fields = POSTPROCESSORS[spec.postprocess](fields, case["message"]) if fields else None
        row = {"id": case["id"], "version": spec.version, "raw_output": resp.text,
               "format_valid": not problems, "problems": "; ".join(problems),
               "input_tokens": resp.input_tokens, "output_tokens": resp.output_tokens, "seconds": resp.latency_s}
        for f in FIELDS:
            pred = fields.get(f) if fields else None
            valid = f not in ALLOWED or pred in ALLOWED[f]
            row[f"pred_{f}"] = pred
            row[f"{f}_ok"] = bool(fields) and valid and pred == case[f]     # invalid values count as wrong
        row["exact_match"] = all(row[f"{f}_ok"] for f in FIELDS)
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(results: pd.DataFrame) -> dict:
    """The numbers we track for every version."""
    tokens_in, tokens_out = results["input_tokens"].mean(), results["output_tokens"].mean()
    return {"category_accuracy": results["category_ok"].mean(), "platform_accuracy": results["platform_ok"].mean(),
            "version_accuracy": results["app_version_ok"].mean(), "exact_match": results["exact_match"].mean(),
            "format_valid_rate": results["format_valid"].mean(),
            "avg_input_tokens": tokens_in, "avg_output_tokens": tokens_out,
            "cost_usd_per_1k": 1000 * api_cost_estimate(tokens_in, tokens_out, "claude-haiku-5-5"),
            "sec_per_msg_local": results["seconds"].mean()}


RESULTS, SPECS = {}, {}


def evaluate(spec: PromptSpec) -> pd.DataFrame:
    """Evaluate a spec, remember results, and print a one-line summary."""
    SPECS[spec.version], RESULTS[spec.version] = spec, run_eval(spec)
    s = summarize(RESULTS[spec.version])
    print(f"{spec.version} ({spec.fingerprint}): category {s['category_accuracy']:.0%} | platform {s['platform_accuracy']:.0%} | "
          f"version {s['version_accuracy']:.0%} | exact {s['exact_match']:.0%} | format-valid {s['format_valid_rate']:.0%} | "
          f"{s['avg_input_tokens']:.0f} input tokens | ${s['cost_usd_per_1k']:.3f}/1k msgs (Haiku prices)")
    return RESULTS[spec.version]

> **Cost note.** `cost_usd_per_1k` uses the *Claude Haiku* prices from the `PRICE_PER_MTOK` table above, so you can budget before switching to an API; the local model itself has no per-token bill. `sec_per_msg_local` is the **amortised** time per message in our CPU batch (it depends heavily on your machine and its load), useful for comparing versions with each other, not as a production latency. For an API you would measure p50/p95 latency of single calls.

---
## 6. Prompt v1: the day-one baseline

Every project starts with a quick prompt somebody wrote in two minutes. We **measure it anyway**: it is the baseline every later version must beat, and it shows which problems are worth fixing first.

In [7]:
v1 = PromptSpec(
    name="feedback_triage", version="v1", changelog="first quick prompt",
    system="You are a helpful assistant.",
    user_template="Classify this app feedback and extract the platform and app version. Answer in JSON.\n\n{message}",
)
res_v1 = evaluate(v1)
res_v1[["id", "raw_output", "problems"]].head(6)

2026-10-21 18:23:27,476 | INFO | Running v1 (0b5f73b98edf) on 15 messages ...


v1 (0b5f73b98edf): category 0% | platform 7% | version 33% | exact 0% | format-valid 0% | 53 input tokens | $0.018/1k msgs (Haiku prices)


,id,raw_output,problems
0,F01,"```json\n{\n ""platform"": ""iOS"",\n ""app_version"": ""5.2.1""\n}\n```","code fence; keys ['app_version', 'platform']; category=None not al..."
1,F02,"```json\n{\n ""platform"": ""web"",\n ""app_version"": ""1.0""\n}\n```","code fence; keys ['app_version', 'platform']; category=None not al..."
2,F03,"```json\n{\n ""platform"": ""iOS"",\n ""app_version"": ""10.2""\n}\n```","code fence; keys ['app_version', 'platform']; category=None not al..."
3,F04,"```json\n{\n ""platform"": ""Android"",\n ""app_version"": ""10.0""\n}\n```","code fence; keys ['app_version', 'platform']; category=None not al..."
4,F05,"```json\n{\n ""platform"": ""iOS"",\n ""app_version"": ""1.2.3""\n}\n```","code fence; keys ['app_version', 'platform']; category=None not al..."
5,F06,"```json\n{\n ""platform"": ""Android"",\n ""app_version"": ""5.3.0""\n}\...","code fence; keys ['app_version', 'platform']; category=None not al..."


Look at the raw outputs, not just the scores. v1 never says *which* categories exist, so the model **does not output a category at all**, wraps the answer in a Markdown code fence, writes platforms as `"iOS"`/`"Android"` instead of our values, and invents version numbers (`"10.2"`, `"1.2.3"`) for messages that contain none. Result: 0% category accuracy and 0% valid format. Nothing about this is surprising **once you look at the outputs**, which is the whole point of measuring early.

---
## 7. Prompt v2: clear instructions + output schema

v2 applies the most basic advice from every guide: **be clear and direct**. Concretely:
- a **role** in the system prompt ("triage assistant for Pocketly"),
- the **allowed categories with definitions**, taken from the labelling guideline, in a fixed **priority order** (this resolves the ambiguous cases the same way the labellers did),
- **field rules** for `platform` and `app_version`,
- the **exact output schema** and "answer with ONLY the JSON object",
- the user message wrapped in **XML tags** (`<feedback>...</feedback>`), which Anthropic recommends to separate instructions from data.

In [8]:
V2_SYSTEM = """You are a triage assistant for Pocketly, a budgeting app. Read one user feedback message and return a JSON object.

Choose "category" with these rules, checked in this order:
1. "account_access": logging in, passwords, SMS or 2FA codes, Face ID, locked accounts.
2. "billing": charges, payments, subscriptions, refunds or prices.
3. "bug": something is broken, crashes, shows wrong data or does not load.
4. "feature_request": the user asks for something new or an improvement.
5. "praise": only positive feedback, nothing to fix and nothing to add.

"platform": "ios" (iPhone, iPad, iOS), "android" (Android, Pixel, Samsung, Galaxy), "web" (website, browser, Chrome), or null if not mentioned.
"app_version": the version number exactly as written (like "5.2.1"), or null if there is none.

Answer with ONLY the JSON object, for example:
{"category": "bug", "platform": null, "app_version": null}"""

v2 = PromptSpec(name="feedback_triage", version="v2", system=V2_SYSTEM,
                changelog="role, category definitions in priority order, field rules, exact schema, XML tags")
res_v2 = evaluate(v2)

2026-10-21 18:23:49,108 | INFO | Running v2 (37f366f847ca) on 15 messages ...


v2 (37f366f847ca): category 73% | platform 40% | version 93% | exact 27% | format-valid 80% | 262 input tokens | $0.036/1k msgs (Haiku prices)


---
## 8. Error analysis: look at the failures, then name them

A single accuracy number does not tell you *what to change*. Error analysis does: read every failure, give each one a short **error type**, and count the types. The most frequent type is the next thing to fix. (This "open coding then counting" approach is what Hamel Husain and others recommend as the core of LLM evaluation.)

In [9]:
def error_types(results: pd.DataFrame, eval_set: pd.DataFrame = EVAL_SET, examples=()) -> pd.DataFrame:
    """Label every wrong field with an error type, automatically where possible."""
    example_text = " ".join(a for _, a in examples)
    rows = []
    for (_, r), (_, gold) in zip(results.iterrows(), eval_set.iterrows()):
        if r["problems"]:
            rows.append((r["id"], "format", r["problems"]))
        if not r["category_ok"]:
            rows.append((r["id"], f"category: {gold['category']} -> {r['pred_category']}", gold["tags"]))
        for f in ["platform", "app_version"]:
            pred, true = r[f"pred_{f}"], gold[f]
            if r[f"{f}_ok"]:
                continue
            if isinstance(pred, str) and pred.lower() in {"null", "none", "unknown", ""}:
                kind = f"{f}: null written as a string"
            elif true is None and pred is not None:
                kind = f"{f}: hallucinated (not in message)"
            elif true is not None and pred is None:
                kind = f"{f}: missed"
            else:
                kind = f"{f}: wrong value"
            if "as a string" not in kind and isinstance(pred, str) and pred in example_text and pred not in gold["message"]:
                kind += " [copied from a few-shot example!]"
            rows.append((r["id"], kind, f"pred={pred!r} true={true!r}"))
    return pd.DataFrame(rows, columns=["id", "error_type", "detail"])


errors_v2 = error_types(res_v2)
print("Error types for v2 (most frequent first):")
print(errors_v2["error_type"].str.replace(r" -> .*", " -> ...", regex=True).value_counts().to_string())
errors_v2

Error types for v2 (most frequent first):
error_type
platform: hallucinated (not in message)    5
format                                     3
platform: null written as a string         3
category: billing -> ...                   2
category: bug -> ...                       1
platform: wrong value                      1
app_version: null written as a string      1
category: account_access -> ...            1


,id,error_type,detail
0,F03,platform: hallucinated (not in message),pred='ios' true=None
1,F04,platform: wrong value,pred='ios' true='android'
2,F05,platform: hallucinated (not in message),pred='web' true=None
3,F06,format,platform='null' not allowed; app_version='null' not allowed
4,F06,category: bug -> account_access,device-name
5,F06,platform: null written as a string,pred='null' true='android'
6,F06,app_version: null written as a string,pred='null' true='5.3.0'
7,F07,platform: hallucinated (not in message),pred='ios' true=None
8,F08,category: billing -> bug,hard: bug caused billing
9,F09,format,platform='null' not allowed


In [10]:
# For the classification field, a confusion matrix shows WHICH classes get mixed up
pd.crosstab(EVAL_SET["category"], res_v2["pred_category"].fillna("(invalid)"),
            rownames=["true category"], colnames=["v2 predicted"])

v2 predicted,account_access,billing,bug,feature_request,praise
true category,,,,,
account_access,2,0,1,0,0
billing,0,1,1,1,0
bug,1,0,2,0,0
feature_request,0,0,0,3,0
praise,0,0,0,0,3


Read the two tables above together with the raw outputs. The questions to ask for each error type are: *is it frequent?*, *is it costly?*, and *is it fixable in the prompt, or somewhere else (labels, code, model)?*

What v2's error analysis says, in order of frequency:
1. **Platform hallucinated**: 5 of 15 messages get a platform although none is named (often `ios` or `web`).
2. **`null` written as a string** (`"null"` instead of JSON `null`): 3× for platform, 1× for app_version. These cause *all* the format failures.
3. **Category errors** (4): two sit exactly on cases we tagged as hard (F08: a bug that caused a wrong charge, which the guideline calls `billing`; F14: Face ID login, which the guideline calls `account_access`), plus F06 and F13.

Category accuracy jumped from 0% to 73% with clear instructions, and the two most frequent error types are about **null handling**, which looks fixable with more precise instructions.

---
## 9. Prompt v3: add few-shot examples

The most common next step is **few-shot prompting**: show a handful of solved examples. The guides recommend examples that are **diverse**, cover the edge cases, and use exactly the output format you want. Our five examples come from a separate pool written by the support team (not from the eval set), one per category, sent as real user/assistant turns.

In [11]:
FEW_SHOT = [
    ("<feedback>\nPaid for Premium on the website but the app still shows the free plan.\n</feedback>",
     '{"category": "billing", "platform": "web", "app_version": null}'),
    ("<feedback>\nThe two-factor code is rejected every time on my iPhone.\n</feedback>",
     '{"category": "account_access", "platform": "ios", "app_version": null}'),
    ("<feedback>\nTransactions list is empty after the 5.1.2 update on my Samsung.\n</feedback>",
     '{"category": "bug", "platform": "android", "app_version": "5.1.2"}'),
    ("<feedback>\nNice app, but please add a home screen widget.\n</feedback>",
     '{"category": "feature_request", "platform": null, "app_version": null}'),
    ("<feedback>\nReally happy with version 5.0.4, it helped me save money!\n</feedback>",
     '{"category": "praise", "platform": null, "app_version": "5.0.4"}'),
]
v3 = PromptSpec(name="feedback_triage", version="v3", system=V2_SYSTEM, examples=FEW_SHOT,
                changelog="v2 + 5 few-shot examples (one per category) as chat turns")
res_v3 = evaluate(v3)
error_types(res_v3, examples=FEW_SHOT)

2026-10-21 18:24:30,840 | INFO | Running v3 (3365d2afe05c) on 15 messages ...


v3 (3365d2afe05c): category 60% | platform 67% | version 93% | exact 40% | format-valid 87% | 515 input tokens | $0.062/1k msgs (Haiku prices)


,id,error_type,detail
0,F02,platform: missed,pred=None true='web'
1,F03,category: billing -> bug,must_pass
2,F04,category: account_access -> bug,"must_pass, device-name"
3,F04,app_version: hallucinated (not in message) [copied from a few-shot...,pred='5.0.4' true=None
4,F06,format,platform='null' not allowed
5,F06,platform: null written as a string,pred='null' true='android'
6,F08,category: billing -> bug,hard: bug caused billing
7,F10,category: praise -> feature_request,hard: version in praise
8,F10,platform: missed,pred=None true='android'
9,F11,platform: missed,pred=None true='web'


Few-shot is a standard tip, but **the measurement decides**, not the tip. Compared with v2:

- **Platform improved** (40% → 67%): the examples show that `null` is a normal answer.
- **Category got worse** (73% → 60%): the model now answers `bug` for five messages, including the must-pass double charge (F03) and the SMS-code login problem (F04).
- F04 got `app_version: "5.0.4"`, a value **copied from a few-shot example**. Small models in particular copy *values* from examples into unrelated answers; our error-type function flags this automatically because it is easy to miss by eye.
- The prompt is now about **twice as long** (≈515 vs. ≈262 input tokens): higher cost and latency on every call, forever.

Section 13 lets the regression gate decide whether v3 may replace v2.

---
## 10. Prompt v4: targeted fixes from the error analysis

Instead of a generic technique, v4 fixes the **specific** error types we counted for v2: platform filled when nothing is named, and `"null"` written as a string. We go back to v2 (no examples) and change only two instructions, so that if something changes we know why. Small, single-purpose changes are the prompt equivalent of small commits.

In [12]:
OLD_FIELD_RULES = """"platform": "ios" (iPhone, iPad, iOS), "android" (Android, Pixel, Samsung, Galaxy), "web" (website, browser, Chrome), or null if not mentioned.
"app_version": the version number exactly as written (like "5.2.1"), or null if there is none."""
NEW_FIELD_RULES = """"platform": "ios" (iPhone, iPad, iOS), "android" (Android, Pixel, Samsung, Galaxy) or "web" (website, browser, Chrome). If the message does not name a device, operating system or browser, "platform" must be null.
"app_version": the version number exactly as written (like "5.2.1"), or null if there is none.
For missing values write null without quotes, never "null" or "unknown"."""
V4_SYSTEM = V2_SYSTEM.replace(OLD_FIELD_RULES, NEW_FIELD_RULES)     # change ONLY the field rules
assert V4_SYSTEM != V2_SYSTEM, "the replacement did not match: check the text"

v4 = PromptSpec(name="feedback_triage", version="v4", system=V4_SYSTEM,
                changelog="v2 + explicit 'platform must be null if not named' + 'null without quotes'")
res_v4 = evaluate(v4)
error_types(res_v4)

2026-10-21 18:25:49,801 | INFO | Running v4 (6edf49866d83) on 15 messages ...


v4 (6edf49866d83): category 67% | platform 93% | version 80% | exact 60% | format-valid 100% | 294 input tokens | $0.039/1k msgs (Haiku prices)


,id,error_type,detail
0,F02,platform: missed,pred=None true='web'
1,F06,category: bug -> feature_request,device-name
2,F06,app_version: missed,pred=None true='5.3.0'
3,F08,category: billing -> bug,hard: bug caused billing
4,F10,category: praise -> feature_request,hard: version in praise
5,F10,app_version: missed,pred=None true='5.3.0'
6,F13,category: billing -> feature_request,easy
7,F14,category: account_access -> bug,hard: login vs bug
8,F14,app_version: missed,pred=None true='5.3.1'


Compared with v2, the targeted rules worked where they were aimed: **platform 40% → 93%** and **valid format 80% → 100%** (no more `"null"` strings). But look at `app_version`: **93% → 80%**. The new "write null" instruction made the model write null **too often**, also for F06, F10 and F14, which do contain a version number. Any change to a prompt can move every metric, which is why we always re-run the **whole** eval set, never just the cases we were trying to fix.

---
## 11. Prompt v5: move what code does better out of the prompt

Look at what the model is asked to do with `app_version`: copy a pattern like `5.3.0` from the text. A one-line regular expression does that **perfectly, instantly and for free**. Good prompt engineering includes knowing when *not* to use the prompt: let the LLM do the part that needs language understanding (category, platform from device names) and let code do the deterministic part.

v5 = v4's prompt + a deterministic post-processing step that sets `app_version` from a regex. Because the prompt text and model are unchanged, the **LLM outputs are reused from the cache** (no new model calls, see the log line), but the spec, and therefore the version fingerprint, is new. The regex itself is code, so it gets unit tests first.

In [13]:
VERSION_IN_TEXT = re.compile(r"(?<![\d.])(\d+\.\d+\.\d+)(?!\.?\d)")    # 5.3.0 / "5.2.1." (not 0.00, not 1.2.3.4)


def regex_app_version(fields: dict, message: str) -> dict:
    """Overwrite app_version with what the message actually contains (or None)."""
    found = VERSION_IN_TEXT.search(message)
    return {**fields, "app_version": found.group(1) if found else None}


POSTPROCESSORS["regex_app_version"] = regex_app_version
# Unit-test the regex on its own before trusting it (it is code, so it gets tests like code)
for text, expected in [("version 5.2.1.", "5.2.1"), ("updated to 5.3.0 on my Pixel", "5.3.0"),
                       ("balance shows 0.00", None), ("charged $9.99 twice", None), ("v5.3.1, you rock", "5.3.1")]:
    assert regex_app_version({}, text)["app_version"] == expected, text
print("regex unit tests PASSED")

v5 = PromptSpec(name="feedback_triage", version="v5", system=V4_SYSTEM, postprocess="regex_app_version",
                changelog="v4 prompt + app_version extracted by regex in code")
res_v5 = evaluate(v5)

2026-10-21 18:26:33,814 | INFO | v5: all 15 model outputs found in the cache (same prompt, same model)


regex unit tests PASSED
v5 (1f841dacbbf1): category 67% | platform 93% | version 100% | exact 60% | format-valid 100% | 294 input tokens | $0.039/1k msgs (Haiku prices)


---
## 12. Compare all versions (with honest uncertainty)

Now the comparison table the product manager asked for. Two extra columns matter in practice: **cost** (input tokens × price, paid on every message forever) and **seconds per message**.

In [14]:
comparison = pd.DataFrame({v: summarize(r) for v, r in RESULTS.items()}).T
comparison.insert(0, "changelog", [SPECS[v].changelog for v in comparison.index])
pct_cols = ["category_accuracy", "platform_accuracy", "version_accuracy", "exact_match", "format_valid_rate"]
styled = comparison.copy()
styled[pct_cols] = (styled[pct_cols] * 100).round(0).astype(int).astype(str) + "%"
styled[["avg_input_tokens", "avg_output_tokens"]] = styled[["avg_input_tokens", "avg_output_tokens"]].astype(float).round(0)
styled["cost_usd_per_1k"] = styled["cost_usd_per_1k"].astype(float).round(4)
styled["sec_per_msg_local"] = styled["sec_per_msg_local"].astype(float).round(2)
styled

,changelog,category_accuracy,platform_accuracy,version_accuracy,exact_match,format_valid_rate,avg_input_tokens,avg_output_tokens,cost_usd_per_1k,sec_per_msg_local
v1,first quick prompt,0%,7%,33%,0%,0%,53.0,25.0,0.0180,1.44
v2,"role, category definitions in priority order, field rules, exact s...",73%,40%,93%,27%,80%,262.0,20.0,0.0364,2.78
v3,v2 + 5 few-shot examples (one per category) as chat turns,60%,67%,93%,40%,87%,515.0,20.0,0.0615,5.26
v4,v2 + explicit 'platform must be null if not named' + 'null without...,67%,93%,80%,60%,100%,294.0,19.0,0.0389,2.93
v5,v4 prompt + app_version extracted by regex in code,67%,93%,100%,60%,100%,294.0,19.0,0.0389,2.93


### How sure are we? Bootstrap confidence intervals

With 15 messages, one message is 6.7 percentage points. Before celebrating a difference, estimate the uncertainty. The **bootstrap** resamples the 15 cases with replacement many times and recomputes the metric; the middle 95% of those values is a confidence interval. For comparing two versions we use a **paired** bootstrap (same resampled cases for both), and also count the cases each version wins and loses, which is often more convincing for stakeholders than any interval.

In [15]:
def bootstrap_ci(values: np.ndarray, rounds: int = CFG.bootstrap_rounds) -> tuple[float, float]:
    idx = rng.integers(0, len(values), size=(rounds, len(values)))
    means = values[idx].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


ci_rows = []
for v, r in RESULTS.items():
    for metric in ["category_ok", "exact_match"]:
        lo, hi = bootstrap_ci(r[metric].to_numpy(float))
        ci_rows.append({"version": v, "metric": metric, "value": r[metric].mean(), "95% CI": f"[{lo:.0%}, {hi:.0%}]"})
print(pd.DataFrame(ci_rows).pivot(index="version", columns="metric", values="95% CI").to_string())


def paired_comparison(a: str, b: str, metric: str) -> str:
    x, y = RESULTS[a][metric].to_numpy(float), RESULTS[b][metric].to_numpy(float)
    lo, hi = bootstrap_ci(y - x)
    wins, losses = int(((y == 1) & (x == 0)).sum()), int(((y == 0) & (x == 1)).sum())
    return f"{b} vs {a} on {metric:12s}: diff {np.mean(y - x):+.0%}, 95% CI [{lo:+.0%}, {hi:+.0%}], {wins} cases fixed, {losses} broken"


print()
for a, b in [("v1", "v2"), ("v2", "v3"), ("v2", "v5")]:
    for metric in ["category_ok", "exact_match"]:
        print(paired_comparison(a, b, metric))

metric  category_ok exact_match
version                        
v1         [0%, 0%]    [0%, 0%]
v2       [47%, 93%]   [7%, 53%]
v3       [33%, 87%]  [20%, 67%]
v4       [40%, 93%]  [33%, 87%]
v5       [40%, 87%]  [33%, 80%]

v2 vs v1 on category_ok : diff +73%, 95% CI [+47%, +93%], 11 cases fixed, 0 broken
v2 vs v1 on exact_match : diff +27%, 95% CI [+7%, +47%], 4 cases fixed, 0 broken
v3 vs v2 on category_ok : diff -13%, 95% CI [-40%, +13%], 1 cases fixed, 3 broken
v3 vs v2 on exact_match : diff +13%, 95% CI [-20%, +47%], 5 cases fixed, 3 broken
v5 vs v2 on category_ok : diff -7%, 95% CI [-20%, +0%], 0 cases fixed, 1 broken
v5 vs v2 on exact_match : diff +33%, 95% CI [-7%, +67%], 7 cases fixed, 2 broken


**How to read this:** an interval that contains 0 means the eval set is too small to be sure about that difference. The v1 → v2 jump is clearly real (the whole interval is far above 0). Even v5's +33 points of exact match over v2 has an interval that reaches below 0, because 15 messages are simply not many; and single-case changes in category accuracy are pure noise at this size. That is why the regression gate below works with **counts of broken cases** and a tolerance, and why the most important long-term investment is a **bigger eval set**, fed by real production failures.

---
## 13. Regression gate: block prompts that make things worse

Before a new version replaces the current one, an automated gate (run in CI on every prompt change, like unit tests for code) compares the candidate with the **reference** version. Our rules:

1. **Must-pass cases**: the candidate must get the category right on every must-pass case.
2. **No field clearly worse**: a field may not lose 2 or more cases compared with the reference (1 case is printed as a WARN for a human: within noise for 15 messages).
3. **Format validity** must not drop by 2 or more cases.
4. **Budget**: cost per 1,000 messages must stay within the budget.

A candidate passes only if every rule passes. Like the "watch the test fail" habit in software, we run the gate on several candidates and check that it blocks the right ones.

In [16]:
def regression_gate(candidate: str, reference: str) -> tuple[bool, list[str]]:
    """Return (passed, report lines) for candidate vs. reference version."""
    cand, ref = RESULTS[candidate], RESULTS[reference]
    lines, passed = [], True
    must = cand.set_index("id").loc[MUST_PASS, "category_ok"]
    ok = bool(must.all())
    lines.append(f"{'PASSED' if ok else 'FAILED'}  must-pass cases correct ({', '.join(i for i in MUST_PASS if not must[i]) or 'all ok'})")
    passed &= ok
    for col in ["category_ok", "platform_ok", "app_version_ok", "format_valid"]:
        r_ok, c_ok = ref[col].to_numpy(bool), cand[col].to_numpy(bool)
        broken = int((r_ok & ~c_ok).sum())                                  # right before, wrong now
        fixed = int((~r_ok & c_ok).sum())
        status = "FAILED" if broken >= 2 else ("WARN  " if broken == 1 else "PASSED")
        lines.append(f"{status}  {col:15s} {fixed} fixed, {broken} broken")
        passed &= broken < 2
    cost = summarize(cand)["cost_usd_per_1k"]
    ok = cost <= CFG.budget_usd_per_1k_msgs
    lines.append(f"{'PASSED' if ok else 'FAILED'}  cost ${cost:.4f}/1k msgs <= budget ${CFG.budget_usd_per_1k_msgs:.2f}")
    passed &= ok
    return passed, lines


GATE_RESULTS = {}
for candidate, reference in [("v2", "v1"), ("v3", "v2"), ("v4", "v2"), ("v5", "v2")]:
    passed, lines = regression_gate(candidate, reference)
    GATE_RESULTS[candidate] = passed
    print(f"Gate: {candidate} vs reference {reference} -> {'PASS' if passed else 'BLOCKED'}")
    print("\n".join("   " + line for line in lines), "\n")

Gate: v2 vs reference v1 -> PASS
   PASSED  must-pass cases correct (all ok)
   PASSED  category_ok     11 fixed, 0 broken
   PASSED  platform_ok     5 fixed, 0 broken
   WARN    app_version_ok  10 fixed, 1 broken
   PASSED  format_valid    12 fixed, 0 broken
   PASSED  cost $0.0364/1k msgs <= budget $0.10 

Gate: v3 vs reference v2 -> BLOCKED
   FAILED  must-pass cases correct (F03, F04)
   FAILED  category_ok     1 fixed, 3 broken
   FAILED  platform_ok     8 fixed, 4 broken
   WARN    app_version_ok  1 fixed, 1 broken
   WARN    format_valid    2 fixed, 1 broken
   PASSED  cost $0.0615/1k msgs <= budget $0.10 

Gate: v4 vs reference v2 -> BLOCKED
   PASSED  must-pass cases correct (all ok)
   WARN    category_ok     0 fixed, 1 broken
   WARN    platform_ok     9 fixed, 1 broken
   FAILED  app_version_ok  0 fixed, 2 broken
   PASSED  format_valid    3 fixed, 0 broken
   PASSED  cost $0.0389/1k msgs <= budget $0.10 

Gate: v5 vs reference v2 -> PASS
   PASSED  must-pass cases correct 

What the gate decided, and why:
- **v2 vs v1: PASS** (one WARN: F06's version became the string `"null"`).
- **v3 vs v2: BLOCKED**: two must-pass cases wrong and three category answers broken. The popular technique made *this* model worse at the most important field.
- **v4 vs v2: BLOCKED**: the platform/format fix is real, but two version answers broke (the over-nulling side effect).
- **v5 vs v2: PASS**, with two WARNs a human should read: one category answer (F10, praise → feature_request) and one platform answer (F02, the web dashboard) got worse.

Read every BLOCKED/PASS decision together with its reasons. This is the same habit as in software: a red CI check is not an insult, it is information. A blocked version is not thrown away either: it is kept in the registry with its results (status `rejected`), so nobody tries the same idea again without knowing what happened.

---
## 14. Prompt registry: versioning, promotion, rollback

Teams keep prompts in **git** (OpenAI's current guide recommends managing production prompts in your code base, with code review and normal deployments) or in a prompt-management tool (Langfuse, LangSmith, PromptLayer, MLflow's prompt registry). Whatever the tool, the registry stores for each version: the full spec, its fingerprint, who changed what and why, the eval results, and a **status** (`draft` → `staging` → `production`, or `rejected` / `archived`). The application loads "whatever is in production", so a **rollback is a one-line change**.

Here the registry is a small JSON file in a temporary folder (in a real repo: `prompts/feedback_triage.json`).

In [17]:
class PromptRegistry:
    """Minimal file-based prompt registry: versions + metrics + status, saved as JSON."""

    def __init__(self, path: Path):
        self.path = path
        self.data = json.loads(path.read_text()) if path.exists() else {"versions": {}, "history": []}

    def _save(self):
        self.path.write_text(json.dumps(self.data, indent=2, default=str))

    def register(self, spec: PromptSpec, metrics: dict, author: str = "prompt-dev"):
        self.data["versions"][spec.version] = {
            "fingerprint": spec.fingerprint, "spec": asdict(spec), "metrics": metrics, "status": "draft",
            "author": author, "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self._save()

    def _log(self, version: str, old: str, new: str, reason: str):
        self.data["history"].append({"version": version, "from": old, "to": new, "reason": reason,
                                     "at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")})

    def set_status(self, version: str, status: str, reason: str, previous_becomes: str = "archived"):
        """Change a version's status. Only ONE version may be in 'staging' and one in 'production'."""
        if status in ("staging", "production"):
            for v, info in self.data["versions"].items():
                if info["status"] == status and v != version:      # the old holder steps down
                    self._log(v, status, previous_becomes, f"replaced by {version}")
                    info["status"] = previous_becomes
        old = self.data["versions"][version]["status"]
        self.data["versions"][version]["status"] = status
        self._log(version, old, status, reason)
        self._save()

    def rollback(self, status: str, to_version: str, reason: str):
        """Put an older version back into `status`; the current holder is marked 'rolled_back'."""
        self.set_status(to_version, status, reason, previous_becomes="rolled_back")

    def get(self, status: str) -> PromptSpec | None:
        """What the application calls at start-up, e.g. registry.get('production')."""
        for info in self.data["versions"].values():
            if info["status"] == status:
                return PromptSpec(**info["spec"])
        return None

    def table(self) -> pd.DataFrame:
        return pd.DataFrame([{"version": v, "status": i["status"], "fingerprint": i["fingerprint"],
                              "category_acc": round(i["metrics"]["category_accuracy"], 2),
                              "exact_match": round(i["metrics"]["exact_match"], 2),
                              "changelog": i["spec"]["changelog"]} for v, i in self.data["versions"].items()])


registry = PromptRegistry(REGISTRY_DIR / "feedback_triage.json")
for version, spec in SPECS.items():
    registry.register(spec, summarize(RESULTS[version]))

# Promotion decisions follow the gate results (never "it looked better to me")
registry.set_status("v2", "staging", "passed gate vs v1; reference for later candidates")
for version in ["v3", "v4", "v5"]:
    if GATE_RESULTS[version]:
        registry.set_status(version, "staging", "passed regression gate vs v2")
    else:
        registry.set_status(version, "rejected", "blocked by regression gate vs v2")
registry.table()

,version,status,fingerprint,category_acc,exact_match,changelog
0,v1,draft,0b5f73b98edf,0.00,0.00,first quick prompt
1,v2,archived,37f366f847ca,0.73,0.27,"role, category definitions in priority order, field rules, exact s..."
2,v3,rejected,3365d2afe05c,0.60,0.40,v2 + 5 few-shot examples (one per category) as chat turns
3,v4,rejected,6edf49866d83,0.67,0.60,v2 + explicit 'platform must be null if not named' + 'null without...
4,v5,staging,1f841dacbbf1,0.67,0.60,v4 prompt + app_version extracted by regex in code


In [18]:
# The application never hard-codes a prompt: it asks the registry at start-up
current = registry.get("staging")
print(f"Staging prompt: {current.version} (fingerprint {current.fingerprint})")

# Rollback drill: suppose the staging version misbehaves in shadow traffic -> ONE call restores the old one
if current.version != "v2":
    registry.rollback("staging", to_version="v2", reason="rollback drill: problem seen in shadow traffic")
    print("After rollback the application gets:", registry.get("staging").version)
    registry.set_status(current.version, "staging", "drill finished: restore")
    print("Restored:", registry.get("staging").version)

print("\nAudit trail (every status change, with a reason):")
print(pd.DataFrame(registry.data["history"])[["version", "from", "to", "reason"]].to_string(index=False))
print("\nRegistry file:", registry.path, f"({registry.path.stat().st_size / 1024:.1f} KB)")

Staging prompt: v5 (fingerprint 1f841dacbbf1)
After rollback the application gets: v2
Restored: v5

Audit trail (every status change, with a reason):
version        from          to                                            reason
     v2       draft     staging passed gate vs v1; reference for later candidates
     v3       draft    rejected                  blocked by regression gate vs v2
     v4       draft    rejected                  blocked by regression gate vs v2
     v2     staging    archived                                    replaced by v5
     v5       draft     staging                      passed regression gate vs v2
     v5     staging rolled_back                                    replaced by v2
     v2    archived     staging    rollback drill: problem seen in shadow traffic
     v2     staging    archived                                    replaced by v5
     v5 rolled_back     staging                           drill finished: restore

Registry file: /tmp/prompt_re

**Why `staging`, not `production`?** Because passing the regression gate only means *"not worse than before"*. Whether a version is *good enough to launch* is a different question, answered by the success criteria from Section 2.

---
## 15. Launch readiness and the official prompt-engineering tips

### Launch readiness: the best version vs. the success criteria

In [19]:
best = max((v for v in RESULTS if registry.data["versions"][v]["status"] == "staging"),
           key=lambda v: summarize(RESULTS[v])["exact_match"])
metrics = summarize(RESULTS[best])
readiness = pd.DataFrame([
    {"metric": m, "target": t, "achieved": round(metrics[m], 3),
     "met": metrics[m] <= t if m == "cost_usd_per_1k" else metrics[m] >= t}
    for m, (t, _) in SUCCESS_CRITERIA.items()])
print(f"Best staging version: {best}")
readiness

Best staging version: v5


,metric,target,achieved,met
0,category_accuracy,0.90,0.667,False
1,platform_accuracy,0.90,0.933,True
2,version_accuracy,0.95,1.000,True
3,format_valid_rate,0.98,1.000,True
4,cost_usd_per_1k,0.10,0.039,True


**The honest conclusion.** The workflow did its job: it found the best version we can build with this setup (v5 meets the platform, version, format and cost targets) and told us precisely where it still falls short: **category accuracy is 67% against a 90% target**, and category is the field that decides routing. With a **0.5-billion-parameter local model** this is not going to be fixed by rewording, so the decision is **do not launch yet**. The next experiment is not "tweak the wording again", it is changing the **model**: set `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` and re-run this notebook. The eval set, the harness, the gate and the registry stay exactly the same, and the comparison will tell you whether the bigger model meets the bar and at what cost. That re-usability is the real product of this notebook.

### The official tips, and what our measurements said about them
The table connects the guidance in Anthropic's and OpenAI's prompt-engineering guides to what we observed. The point is not that a tip is "wrong": tips are **hypotheses**, and the eval set is how you test them for *your* task and model.

In [20]:
tips = pd.DataFrame([
    ("Define success criteria and tests first", "Anthropic (prompt engineering overview), OpenAI (evals)", "Sections 2-3", "made every later decision measurable"),
    ("Be clear and direct; give a role", "Anthropic best practices, OpenAI", "v2", "biggest jump in this notebook: category 0% -> 73%"),
    ("Specify the output format exactly", "both", "v2, v4", "format errors went down when the schema and 'null without quotes' were explicit"),
    ("Use XML tags / delimiters for data", "Anthropic; OpenAI (Markdown + XML sections)", "v2+", "keeps message text separate from instructions (also helps against injection)"),
    ("Few-shot examples (diverse, as message pairs)", "both", "v3", "here: platform better, category worse, a copied value, 2x tokens -> blocked"),
    ("Change one thing at a time; re-run all tests", "OpenAI (evals when iterating or upgrading models)", "v4", "the null rule fixed platform but broke app_version"),
    ("Use code for deterministic sub-tasks", "OpenAI ('use external tools'), Anthropic (chain/split tasks)", "v5", "regex for app_version: free, instant, exact"),
    ("Pin model snapshots, version prompts in code", "OpenAI", "Section 14", "fingerprint covers model + params + post-processing"),
    ("Reusable content first (prompt caching)", "OpenAI; Anthropic prompt caching", "all", "long shared system prompts get cheaper with provider caching"),
], columns=["tip", "source", "where here", "what we learned"])
tips

,tip,source,where here,what we learned
0,Define success criteria and tests first,"Anthropic (prompt engineering overview), OpenAI (evals)",Sections 2-3,made every later decision measurable
1,Be clear and direct; give a role,"Anthropic best practices, OpenAI",v2,biggest jump in this notebook: category 0% -> 73%
2,Specify the output format exactly,both,"v2, v4",format errors went down when the schema and 'null without quotes' ...
3,Use XML tags / delimiters for data,Anthropic; OpenAI (Markdown + XML sections),v2+,keeps message text separate from instructions (also helps against ...
4,"Few-shot examples (diverse, as message pairs)",both,v3,"here: platform better, category worse, a copied value, 2x tokens -..."
5,Change one thing at a time; re-run all tests,OpenAI (evals when iterating or upgrading models),v4,the null rule fixed platform but broke app_version
6,Use code for deterministic sub-tasks,"OpenAI ('use external tools'), Anthropic (chain/split tasks)",v5,"regex for app_version: free, instant, exact"
7,"Pin model snapshots, version prompts in code",OpenAI,Section 14,fingerprint covers model + params + post-processing
8,Reusable content first (prompt caching),OpenAI; Anthropic prompt caching,all,long shared system prompts get cheaper with provider caching


---
## 16. Summary, exercises and resources

| Practice | What we did | Where it is used in industry |
|---|---|---|
| Success criteria first | Targets per metric agreed before prompting | Every LLM product spec |
| Labelling guideline | Priority rules for ambiguous cases | Data labelling, eval sets |
| Labelled eval set with tags + must-pass cases | 15 cases incl. hard ones, separate few-shot pool | promptfoo, OpenAI Evals, LangSmith datasets |
| Prompt = versioned spec with fingerprint | text + examples + model + params + post-processing | Prompt registries, git |
| Strict format validation + field scoring | Parse tolerantly, validate strictly | Structured-output pipelines |
| Error analysis with error types | Counted, then fixed the most frequent | "Look at your data" (Hamel Husain) |
| One change per version | v2 → v3 / v4 / v5 | Small commits, A/B discipline |
| Bootstrap CIs, paired wins/losses | Honest uncertainty with small eval sets | Experiment reviews |
| Regression gate in CI | Must-pass + per-field "broken cases" + budget | CI/CD for prompts |
| Registry with status + audit trail + rollback | draft → staging → production / rejected | Langfuse, LangSmith, MLflow, PromptLayer |

### Practice exercises
1. Add five new hard cases to `EVAL_SET` (e.g. a message in German, sarcasm, two problems in one message). Re-run: which version is most robust?
2. Write v6 that combines v4's rules with **two** carefully chosen few-shot examples that contain no version numbers. Does the gate pass it against v5?
3. Make the gate stricter for `category` (any broken case fails) and looser for `platform`. Argue which choice the product manager would prefer, and why.
4. If you have an API key: set it, re-run the notebook and compare the launch-readiness table for the local model and the API model. What does each extra percentage point of accuracy cost per 1,000 messages?
5. Add a `latency_p95` criterion and measure single (non-batched) calls for the best version.

### Resources
- Anthropic, Prompt engineering overview (success criteria and empirical tests come first): https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview
- Anthropic, Prompting best practices (clarity, examples, XML tags, roles, chaining): https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices
- Anthropic, Define success criteria and build evaluations: https://platform.claude.com/docs/en/test-and-evaluate/develop-tests
- Anthropic, Batch processing (cheaper offline evaluation runs): https://platform.claude.com/docs/en/build-with-claude/batch-processing
- OpenAI, Prompt engineering guide (message roles, few-shot, evals, pinning snapshots, prompts in code): https://developers.openai.com/api/docs/guides/prompt-engineering
- Anthropic's interactive prompt-engineering tutorial: https://github.com/anthropics/prompt-eng-interactive-tutorial
- Hamel Husain, "Your AI Product Needs Evals" (unit tests, error analysis, looking at data): https://hamel.dev/blog/posts/evals/
- promptfoo (open-source prompt testing and regression checks in CI): https://github.com/promptfoo/promptfoo
- OpenAI Evals (framework and registry of evals): https://github.com/openai/evals

### Related notebooks in this repo
- `01_basics.ipynb`, `02_techniques.ipynb`, `03_advanced.ipynb`: the techniques we tested here (roles, few-shot, structured outputs)
- `04_adversarial.ipynb`: prompt injection, which belongs in every eval set as a must-pass category
- `../08_LLM/20_industry_llm_application_patterns.ipynb`: validation, retries, caching and fallbacks around the prompt
- `../08_LLM/08_evaluation/01_llm_evaluation.ipynb`: LLM evaluation metrics and benchmarks
- `../19_AI_Ethics_and_Safety/09_industry_ai_incident_response_and_red_teaming.ipynb`: red-team suites and regression gates for a safety filter